In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

In [2]:
import pickle

# Import the data
data = pd.read_csv('Churn_Modelling.csv')

# Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encode categorical variables
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])

# One Hot Encoding for Geography
data['Geography'].unique()

from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)

geo_encoder = onehot_encoder.fit_transform(data[['Geography']])

geo_encoded = pd.DataFrame(geo_encoder, columns = onehot_encoder.get_feature_names_out(['Geography']))

# Combine one hot encoded columns with the original dataframe
data = pd.concat([data.drop('Geography', axis = 1), geo_encoded], axis = 1)

# Divide the dataset into independent and dependent features
X = data.drop('Exited', axis = 1)
y = data['Exited']

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scale these features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Save the encoder
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)

# Save the onehot encoder
with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder, file)

# Save the scaler
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [3]:
# Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation = 'relu', input_shape = (x_train.shape[1],)))
    
    for _ in range(layers-1):
        model.add(Dense(neurons, activation = 'relu'))
        
    model.add(Dense(1, activation = 'sigmoid'))
    model.compile(optimizer='adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    
    return model

In [4]:
# Create a Keras classifier
model = KerasClassifier(layers = 1, neurons = 32, build_fn = create_model, epochs = 50, batch_size = 10, verbose = 0)

In [5]:
#  Define the grid search parameters
param_grid = {
    'neurons' : [16, 32, 64, 128],
    'layers' : [1, 2],
    'epochs' : [50, 100]
}

In [6]:
# Perform
grid = GridSearchCV(estimator = model, param_grid = param_grid, n_jobs = -1, cv = 3, verbose = 1)
grid_result = grid.fit(x_train, y_train)

# Print the best parameters
print('Best %f using %s' % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


d:\Udemy\Gen AI\Projects\ANN Classification\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)






Best 0.856750 using {'epochs': 50, 'layers': 1, 'neurons': 16}
